# Compare payload vs output

Validates that a processed result file correctly reflects its input payload:
- Same plans (FileNames in / planids out)
- Reasonable row counts per plan
- Which benefits got filled vs missed per plan

**Update `PAYLOAD_PATH` and `OUTPUT_PATH` below.**


In [ ]:
import json
import re
from pathlib import Path
from collections import defaultdict
import pandas as pd

# ── Config ────────────────────────────────────────────────────────────────────
PAYLOAD_PATH = Path(r'C:\\Users\\i40212888\\Downloads\\210-20260511T174006Z_payload.json')
OUTPUT_PATH  = Path(r'C:\\Users\\i40212888\\Downloads\\210-20260511T174006Z_output.json')

# Same carrier prefix logic as build_benefits.py — keeps the planid math in sync
CARRIER_PREFIX_MAP = {
    'humana':   'MOM', 'aetna':    'AETNA', 'uhc':      'UHC',
    'anthem':   'ANTHEM', 'cigna': 'CIGNA', 'wellcare': 'WELLCARE',
    'kaiser':   'KAISER', 'bcbs':  'BCBS',
}

def detect_carrier(file_name: str) -> str:
    if not file_name:
        return 'UNKNOWN'
    fn = file_name.lower()
    matches = [(k, v) for k, v in CARRIER_PREFIX_MAP.items() if fn.startswith(k)]
    if matches:
        matches.sort(key=lambda kv: -len(kv[0]))
        return matches[0][1]
    return file_name.split('_')[0].upper() if '_' in file_name else file_name.upper()

def filename_to_planid(file_name: str) -> str:
    carrier = detect_carrier(file_name)
    remainder = file_name[file_name.index('_') + 1:] if '_' in file_name else file_name
    parts = remainder.split('-')
    contract = parts[0] if len(parts) > 0 else ''
    plan = parts[1] if len(parts) > 1 else ''
    seg = int(parts[2]) if len(parts) > 2 and parts[2].isdigit() else 0
    return f'{carrier}_{contract}_{plan}_{seg}'


## 1. Load both files

In [ ]:
# Load payload (input)
payload_raw = json.loads(PAYLOAD_PATH.read_text(encoding='utf-8'))
if isinstance(payload_raw, dict) and 'pbp' in payload_raw:
    payload_rows = payload_raw['pbp']
else:
    payload_rows = payload_raw

# Load output
output_rows = json.loads(OUTPUT_PATH.read_text(encoding='utf-8'))
# Output may be wrapped in {results: [...]} or be a bare list
if isinstance(output_rows, dict) and 'results' in output_rows:
    output_rows = output_rows['results']

print(f'Payload: {len(payload_rows):,} PBP rows from {PAYLOAD_PATH.name}')
print(f'Output:  {len(output_rows):,} benefit rows from {OUTPUT_PATH.name}')


## 2. Plan-level comparison

Did every input plan produce output rows?

In [ ]:
# Count input rows per FileName
input_plans = defaultdict(int)
for r in payload_rows:
    fn = r.get('FileName')
    if fn:
        input_plans[fn.strip()] += 1

# Map each input FileName → expected planid
expected_planids = {fn: filename_to_planid(fn) for fn in input_plans}

# Count output rows per planid
output_plans = defaultdict(int)
for r in output_rows:
    pid = r.get('planid')
    if pid:
        output_plans[pid] += 1

# Build comparison table
rows = []
for fn, n_input in sorted(input_plans.items()):
    pid = expected_planids[fn]
    n_output = output_plans.get(pid, 0)
    rows.append({
        'FileName':     fn,
        'planid':       pid,
        'input_rows':   n_input,
        'output_rows':  n_output,
        'got_output':   '✓' if n_output > 0 else '✗ MISSING',
    })

# Add any planids in output that weren't matched to an input
unexpected = set(output_plans) - set(expected_planids.values())
for pid in sorted(unexpected):
    rows.append({
        'FileName':    '(none — unexpected planid in output)',
        'planid':      pid,
        'input_rows':  0,
        'output_rows': output_plans[pid],
        'got_output':  '⚠ EXTRA',
    })

cmp_df = pd.DataFrame(rows)
print(f'Input plans:  {len(input_plans)}')
print(f'Output plans: {len(output_plans)}')
print()
display(cmp_df)


## 3. Headline result

In [ ]:
missing = cmp_df[cmp_df['got_output'].str.contains('MISSING')]
extra   = cmp_df[cmp_df['got_output'].str.contains('EXTRA')]

if len(missing) == 0 and len(extra) == 0:
    print(f'✓ All {len(input_plans)} input plans produced output rows. No unexpected plans.')
elif len(missing) > 0:
    print(f'✗ {len(missing)} input plan(s) produced NO output:')
    for _, r in missing.iterrows():
        print(f'   - {r["FileName"]} → expected planid {r["planid"]}')

if len(extra) > 0:
    print(f'⚠ {len(extra)} planid(s) in output not matched to any input FileName')
    for _, r in extra.iterrows():
        print(f'   - {r["planid"]}')


## 4. Per-benefit coverage matrix

For each plan, which of the ~47 expected benefits were produced? Any plan with many empty cells likely had a per-benefit LLM call fail or genuinely doesn't offer that benefit.

In [ ]:
# Expected benefit IDs (from BENEFIT_TARGETS in build_benefits.py)
EXPECTED_BENEFITS = [
    '600', '610', '611', '614', '615', '616', '620',
    '700', '710', '711', '730', '740', '755', '760',
    '800', '810', '820',
    '900', '910', '911', '920', '930', '940', '950', '960', '970',
    '981', '982', '990',
    '1000', '1020', '1030',
    '1050', '1060',
    '1200', '1300', '1301', '1400', '1500',
    '1610', '1700', '1800', '1900',
    '2100', '2110', '2111', '2112',
]

# Pivot: rows = planid, cols = benefitid, values = count of rows for that pair
out_df = pd.DataFrame(output_rows)
if not out_df.empty:
    out_df['benefitid'] = out_df['benefitid'].astype(str)
    matrix = (
        out_df
        .groupby(['planid', 'benefitid'])
        .size()
        .unstack(fill_value=0)
        .reindex(columns=EXPECTED_BENEFITS, fill_value=0)
    )

    # Show with checkmarks for readability
    pretty = matrix.copy()
    for c in pretty.columns:
        pretty[c] = pretty[c].apply(lambda n: '·' if n == 0 else str(n))
    print(f'Coverage matrix ({len(matrix)} plans × {len(EXPECTED_BENEFITS)} benefits):\n')
    print('  · = no rows produced for this benefit\n')
    display(pretty)
else:
    print('Output is empty — nothing to compare')


## 5. Per-plan summary stats

In [ ]:
if not out_df.empty:
    per_plan = (
        out_df
        .groupby('planid')
        .agg(
            output_rows=('benefitid', 'size'),
            distinct_benefits=('benefitid', 'nunique'),
        )
    )
    per_plan['benefits_missing'] = len(EXPECTED_BENEFITS) - per_plan['distinct_benefits']
    per_plan['pct_complete'] = (per_plan['distinct_benefits'] / len(EXPECTED_BENEFITS) * 100).round(1)
    per_plan = per_plan.sort_values('output_rows', ascending=False)
    display(per_plan)

    # Spot the outliers
    median_rows = per_plan['output_rows'].median()
    weak = per_plan[per_plan['output_rows'] < median_rows * 0.5]
    if len(weak) > 0:
        print(f'\n⚠ {len(weak)} plan(s) have <50% of the median row count — '
              f'likely had failed LLM calls or sparse data:')
        for pid, row in weak.iterrows():
            print(f'   - {pid}: {row["output_rows"]} rows, '
                  f'{row["distinct_benefits"]} of {len(EXPECTED_BENEFITS)} benefits')


## 6. Sample output rows

Quick eyeball check of what came out.

In [ ]:
if not out_df.empty:
    cols_to_show = [c for c in ['planid','benefitid','benefitname','serviceTypeDesc','tinyDescription']
                    if c in out_df.columns]
    display(out_df[cols_to_show].head(30))


## 7. Optional — save the comparison report

In [ ]:
out_name = OUTPUT_PATH.stem + '_comparison.csv'
cmp_df.to_csv(out_name, index=False)
print(f'Saved plan comparison to {out_name}')

if not out_df.empty:
    matrix_name = OUTPUT_PATH.stem + '_coverage_matrix.csv'
    matrix.to_csv(matrix_name)
    print(f'Saved coverage matrix to {matrix_name}')
